# Particionamiento y preprocesamiento para modelado

Este notebook toma como entrada el archivo `dataset_unificado_preparado.parquet`, generado en la etapa de unificación, limpieza y EDA. Su objetivo es definir las particiones de entrenamiento, validación y prueba, establecer las variables predictoras y objetivo, construir muestras manejables para experimentación y preparar las transformaciones que serán usadas posteriormente por los modelos de detección.

In [1]:
from pathlib import Path
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import duckdb
import pyarrow.parquet as pq
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SECOND_RANDOM_STATE = 2026
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [2]:
# Rutas del proyecto
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data"
PROCESSED_PATH = DATA_PATH / "processed"
INTERIM_PATH = DATA_PATH / "interim"
RESULTS_PATH = PROJECT_ROOT / "results"
MODELING_RESULTS_PATH = RESULTS_PATH / "modeling"
PREPROCESSING_RESULTS_PATH = RESULTS_PATH / "preprocessing"
MODELS_PATH = PROJECT_ROOT / "models"

for path in [DATA_PATH, PROCESSED_PATH, INTERIM_PATH, RESULTS_PATH, MODELING_RESULTS_PATH, PREPROCESSING_RESULTS_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

prepared_parquet_path = (PROCESSED_PATH / "dataset_unificado_preparado.parquet")
prepared_parquet_path

WindowsPath('C:/Users/Laura/Documents/TrabajoGrado2026/TG2026/data/processed/dataset_unificado_preparado.parquet')

In [3]:
# Validación de existencia del dataset preparado
if not prepared_parquet_path.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo preparado en: {prepared_parquet_path}"
    )
print("Dataset preparado encontrado correctamente.")
print(f"Ruta: {prepared_parquet_path}")

Dataset preparado encontrado correctamente.
Ruta: C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\data\processed\dataset_unificado_preparado.parquet


In [4]:
# Configuración de DuckDB
DUCKDB_TEMP_PATH = INTERIM_PATH / "duckdb_tmp"
DUCKDB_TEMP_PATH.mkdir(parents=True, exist_ok=True)
parquet_sql_path = (prepared_parquet_path
    .resolve()
    .as_posix()
    .replace("'", "''")
)
temp_sql_path = (DUCKDB_TEMP_PATH
    .resolve()
    .as_posix()
    .replace("'", "''")
)
con = duckdb.connect()
con.execute("SET memory_limit = '6GB'")
con.execute("SET threads = 6")
con.execute(f"SET temp_directory = '{temp_sql_path}'")
print("DuckDB configurado correctamente.")

DuckDB configurado correctamente.


In [5]:
# Metadatos del dataset preparado
prepared_file = pq.ParquetFile(prepared_parquet_path)
n_rows = prepared_file.metadata.num_rows
n_columns = len(prepared_file.schema_arrow.names)
prepared_columns = prepared_file.schema_arrow.names
prepared_schema = prepared_file.schema_arrow
print(f"Filas: {n_rows:,}")
print(f"Columnas: {n_columns}")
prepared_columns

Filas: 22,576,261
Columnas: 26


['td',
 'sa',
 'da',
 'sp',
 'dp',
 'pr',
 'tcp_flag_fin',
 'tcp_flag_syn',
 'tcp_flag_rst',
 'tcp_flag_psh',
 'tcp_flag_ack',
 'tcp_flag_urg',
 'tcp_flag_ece',
 'tcp_flag_cwr',
 'stos',
 'ipkt',
 'ibyt',
 'in_if',
 'out_if',
 'sas',
 'das',
 'exp',
 'attack_t',
 'attack_a',
 'dataset_name',
 'attack_family']

In [6]:
# Esquema del dataset preparado
schema_rows = []
for field in prepared_schema:
    schema_rows.append({"variable": field.name, "tipo_arrow": str(field.type)})
df_schema_prepared = pd.DataFrame(schema_rows)
df_schema_prepared

,variable,tipo_arrow
0,td,float
1,sa,string
2,da,string
3,sp,int32
4,dp,int32
5,pr,string
6,tcp_flag_fin,int8
7,tcp_flag_syn,int8
8,tcp_flag_rst,int8
9,tcp_flag_psh,int8


In [7]:
# Muestra pequeña del dataset preparado
pd.set_option('display.max_columns', None)
df_sample_preview = con.execute(f"""
SELECT *
FROM read_parquet('{parquet_sql_path}')
LIMIT 10
""").df()
df_sample_preview

,td,sa,da,sp,dp,pr,tcp_flag_fin,tcp_flag_syn,tcp_flag_rst,tcp_flag_psh,tcp_flag_ack,tcp_flag_urg,tcp_flag_ece,tcp_flag_cwr,stos,ipkt,ibyt,in_if,out_if,sas,das,exp,attack_t,attack_a,dataset_name,attack_family
0,0.280000,193.219.74.172,83.169.5.121,59758,3306,TCP,1,0,0,1,1,0,0,0,8,10,1080,762,556,2847,8972,1,none,0,BLASTER_WORM_v2,worm
1,4.730000,184.73.156.247,193.219.75.85,443,58895,TCP,1,0,0,0,1,0,0,0,0,10,7300,707,588,14618,2847,1,none,0,BLASTER_WORM_v2,worm
2,0.000000,83.171.40.3,80.82.77.132,6666,56667,TCP,0,0,1,0,1,0,0,0,0,5,200,922,556,2847,202425,1,none,0,BLASTER_WORM_v2,worm
3,19.290001,95.108.213.17,158.129.192.240,62443,80,TCP,1,1,0,0,1,0,0,0,164,20,1080,556,558,13238,2847,1,none,0,BLASTER_WORM_v2,worm
4,1.820000,124.115.207.226,158.129.192.178,56637,33896,TCP,0,0,1,1,1,0,0,0,0,10,825,707,558,4134,2847,1,none,0,BLASTER_WORM_v2,worm
5,0.000000,80.82.77.132,158.129.192.215,56667,33389,TCP,0,0,1,0,0,0,0,0,0,5,200,707,558,202425,2847,1,none,0,BLASTER_WORM_v2,worm
6,0.930000,52.97.186.114,193.219.75.225,143,52165,TCP,0,0,1,1,1,0,0,0,164,30,3230,556,592,8075,2847,1,none,0,BLASTER_WORM_v2,worm
7,0.280000,64.179.209.3,193.219.165.4,53,54670,TCP,1,1,0,0,1,0,0,0,128,10,560,556,607,40224,2847,1,none,0,BLASTER_WORM_v2,worm
8,0.310000,193.219.163.162,47.246.2.227,58569,443,TCP,0,1,1,0,1,0,0,0,0,20,900,589,556,2847,24429,1,none,0,BLASTER_WORM_v2,worm
9,2.880000,193.219.36.204,180.244.234.201,22,33741,TCP,1,0,0,1,1,0,0,0,0,15,860,922,556,2847,7713,1,none,0,BLASTER_WORM_v2,worm


In [8]:
# Distribución global de la etiqueta binaria
query_binary_distribution = f"""
SELECT
    attack_a,
    COUNT(*) AS n,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER() AS porcentaje
FROM read_parquet('{parquet_sql_path}')
GROUP BY attack_a
ORDER BY attack_a
"""
df_binary_distribution = con.execute(query_binary_distribution).df()
df_binary_distribution

,attack_a,n,porcentaje
0,0,21107443,93.493971
1,1,1468818,6.506029


In [9]:
# Distribución por tipo de tráfico
query_attack_type_distribution = f"""
SELECT
    CASE
        WHEN attack_a = 0 THEN 'normal'
        ELSE attack_t
    END AS clase,
    COUNT(*) AS n,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER() AS porcentaje
FROM read_parquet('{parquet_sql_path}')
GROUP BY clase
ORDER BY n DESC
"""
df_attack_type_distribution = con.execute(query_attack_type_distribution).df()
df_attack_type_distribution

,clase,n,porcentaje
0,normal,21107443,93.493971
1,tcp_red_w,1255702,5.562046
2,udp_f,93583,0.414519
3,icmp_smf,59479,0.263458
4,tcp_w32_w,24291,0.107595
5,http_f,22959,0.101695
6,icmp_f,11628,0.051505
7,udp_reaper_w,1176,0.005209


## 1. Definición de variables

In [11]:
# Variable objetivo para detección binaria:
# 0 = tráfico normal, 1 = ataque
binary_target_column = "attack_a"

# Variable objetivo para identificación del tipo de tráfico o ataque
multiclass_target_column = "attack_t"

# Variables auxiliares: útiles para estratificación, análisis y evaluación, pero no deben usarse como predictoras
auxiliary_columns = ["dataset_name", "attack_family"]

target_columns = [binary_target_column, multiclass_target_column]

non_predictor_columns = (target_columns + auxiliary_columns)

predictor_candidate_columns = [
    col
    for col in prepared_columns
    if col not in non_predictor_columns
]
print(f"Número de columnas totales: {len(prepared_columns)}")
print(f"Número de predictoras candidatas: {len(predictor_candidate_columns)}")
print(f"Número de variables objetivo: {len(target_columns)}")
print(f"Número de variables auxiliares: {len(auxiliary_columns)}")

Número de columnas totales: 26
Número de predictoras candidatas: 22
Número de variables objetivo: 2
Número de variables auxiliares: 2


In [12]:
# Clasificación semántica inicial de variables

# Variables numéricas continuas o de volumen
numeric_continuous_columns = ["td", "ipkt", "ibyt"]

# Variables binarias ya normalizadas
binary_flag_columns = ["tcp_flag_fin", "tcp_flag_syn", "tcp_flag_rst", "tcp_flag_psh", "tcp_flag_ack", "tcp_flag_urg", "tcp_flag_ece", "tcp_flag_cwr"]

# Variables asociadas a puertos, aunque están codificadas como enteros, su valor no debe interpretarse como una magnitud continua ordinaria
port_columns = ["sp", "dp"]

# Variables categóricas o de contexto de red
network_entity_columns = ["sa", "da", "sas", "das", "in_if", "out_if"]

# Variables categóricas de baja o media cardinalidad esperada
categorical_context_columns = ["pr", "stos", "exp"]

# Variables candidatas a transformación log1p, no se incluyen puertos porque el valor del puerto no representa una escala continua.
log_transform_columns = ["td", "ipkt", "ibyt"]

mencionar que algunas de las variables que dijiste que son categóricas en el código salen como "int.."

### Revisión de cardinalidad

In [13]:
# Cardinalidad de variables predictoras candidatas
cardinality_expressions = []
for col in predictor_candidate_columns:
    cardinality_expressions.append(f"COUNT(DISTINCT {col}) AS n_unique_{col}")

query_cardinality = f"""
SELECT
    {", ".join(cardinality_expressions)}
FROM read_parquet('{parquet_sql_path}')
"""
df_cardinality_wide = con.execute(query_cardinality).df()
df_cardinality = (df_cardinality_wide
    .T
    .reset_index()
)
df_cardinality.columns = ["variable", "n_valores_unicos"]
df_cardinality["variable"] = (df_cardinality["variable"]
    .str.replace("n_unique_", "", regex=False)
)
df_cardinality = (df_cardinality
    .sort_values("n_valores_unicos", ascending=False)
    .reset_index(drop=True)
)
df_cardinality

,variable,n_valores_unicos
0,sa,3853334
1,da,648529
2,ibyt,301892
3,dp,65534
4,sp,65530
5,sas,22391
6,das,18039
7,ipkt,14091
8,td,12504
9,stos,134


In [14]:
# Clasificación inicial por cardinalidad
def classify_cardinality(n_unique):
    if n_unique <= 20:
        return "baja"
    elif n_unique <= 500:
        return "media"
    else:
        return "alta"

df_cardinality["cardinalidad"] = (df_cardinality["n_valores_unicos"]
    .apply(classify_cardinality)
)
df_cardinality

,variable,n_valores_unicos,cardinalidad
0,sa,3853334,alta
1,da,648529,alta
2,ibyt,301892,alta
3,dp,65534,alta
4,sp,65530,alta
5,sas,22391,alta
6,das,18039,alta
7,ipkt,14091,alta
8,td,12504,alta
9,stos,134,media


In [15]:
# Definición de variables categóricas según cardinalidad
# Baja cardinalidad: candidatas a one-hot encoding tradicional
low_cardinality_categorical_columns = ["pr", "exp"]

# Media cardinalidad: pueden codificarse con one-hot agrupando categorías raras, o mediante codificación por frecuencia según el modelo.
medium_cardinality_categorical_columns = ["stos", "in_if", "out_if"]

# Alta cardinalidad: no conviene aplicar one-hot completo. Requieren estrategias controladas como hashing, frecuencia, agrupación de categorías raras o exclusión en experimentos comparativos.
high_cardinality_categorical_columns = ["sa", "da", "sp", "dp", "sas", "das"]

Las variables predictoras se agruparon de la siguiente manera: Las variables `td`, `ipkt` e `ibyt` representan duración y volumen del flujo, por lo que podrán transformarse mediante `log1p` y escalarse en los modelos sensibles a la escala. Las banderas TCP normalizadas se conservarán como variables binarias. Las variables categóricas de baja cardinalidad, como `pr` y `exp`, podrán codificarse mediante one-hot encoding. Las variables de media y alta cardinalidad, como IP, puertos, sistemas autónomos e interfaces, requieren estrategias más controladas de codificación para evitar matrices excesivamente grandes o aprendizaje excesivamente dependiente de valores específicos de infraestructura.

In [19]:
# Tabla resumen de variables para modelado
variable_modeling_rows = []
for col in numeric_continuous_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Predictora candidata",
        "grupo": "Numérica continua / volumen",
        "tratamiento_base": "log1p opcional y escalamiento según modelo"
    })

for col in binary_flag_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Predictora candidata",
        "grupo": "Binaria TCP",
        "tratamiento_base": "Conservar como 0/1"
    })

for col in low_cardinality_categorical_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Predictora candidata",
        "grupo": "Categórica de baja cardinalidad",
        "tratamiento_base": "One-hot encoding"
    })

for col in medium_cardinality_categorical_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Predictora candidata",
        "grupo": "Categórica de media cardinalidad",
        "tratamiento_base": "One-hot con agrupación o codificación por frecuencia"
    })

for col in high_cardinality_categorical_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Predictora candidata",
        "grupo": "Categórica de alta cardinalidad",
        "tratamiento_base": "Hashing, frecuencia, prefijos o experimento sin la variable"
    })

for col in target_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Variable objetivo",
        "grupo": "Etiqueta",
        "tratamiento_base": "No usar como predictora"
    })

for col in auxiliary_columns:
    variable_modeling_rows.append({
        "variable": col,
        "rol": "Auxiliar",
        "grupo": "Estratificación y evaluación",
        "tratamiento_base": "No usar como predictora"
    })

df_modeling_variable_summary = (
    pd.DataFrame(variable_modeling_rows)
    .sort_values(["rol", "grupo", "variable"])
    .reset_index(drop=True)
)
df_modeling_variable_summary

,variable,rol,grupo,tratamiento_base
0,attack_family,Auxiliar,Estratificación y evaluación,No usar como predictora
1,dataset_name,Auxiliar,Estratificación y evaluación,No usar como predictora
2,tcp_flag_ack,Predictora candidata,Binaria TCP,Conservar como 0/1
3,tcp_flag_cwr,Predictora candidata,Binaria TCP,Conservar como 0/1
4,tcp_flag_ece,Predictora candidata,Binaria TCP,Conservar como 0/1
5,tcp_flag_fin,Predictora candidata,Binaria TCP,Conservar como 0/1
6,tcp_flag_psh,Predictora candidata,Binaria TCP,Conservar como 0/1
7,tcp_flag_rst,Predictora candidata,Binaria TCP,Conservar como 0/1
8,tcp_flag_syn,Predictora candidata,Binaria TCP,Conservar como 0/1
9,tcp_flag_urg,Predictora candidata,Binaria TCP,Conservar como 0/1


In [17]:
# Guardar resumen de variables
variable_summary_path = (PREPROCESSING_RESULTS_PATH / "resumen_variables_modelado.csv")
cardinality_summary_path = (PREPROCESSING_RESULTS_PATH / "cardinalidad_predictoras.csv")
df_modeling_variable_summary.to_csv(variable_summary_path, index=False, encoding="utf-8-sig")
df_cardinality.to_csv(cardinality_summary_path, index=False, encoding="utf-8-sig")
print("Resumen de variables guardado en:")
print(variable_summary_path)
print("\nResumen de cardinalidad guardado en:")
print(cardinality_summary_path)

Resumen de variables guardado en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\preprocessing\resumen_variables_modelado.csv

Resumen de cardinalidad guardado en:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\results\preprocessing\cardinalidad_predictoras.csv


## 2. Particionamiento train/validación/test

En esta sección se define una única partición base del conjunto preparado en entrenamiento, validación y prueba. Esta partición será compartida por los modelos de detección binaria y por los modelos de clasificación del tipo de ataque. Para conservar la representación de los distintos subconjuntos y clases, la estratificación se realiza combinando el dataset de origen con la clase de análisis, de esta forma, el tráfico normal de cada subconjunto y cada tipo de ataque mantienen presencia en las tres particiones.

### Definir proporciones y variable de estratificación

In [20]:
TRAIN_SIZE = 0.70
VALID_SIZE = 0.15
TEST_SIZE = 0.15

if not np.isclose(TRAIN_SIZE + VALID_SIZE + TEST_SIZE, 1.0):
    raise ValueError("Las proporciones de split no suman 1.")

SPLIT_ORDER = ["train", "valid", "test"]
CLASS_ORDER = ["normal", "icmp_f", "icmp_smf", "udp_f", "http_f", "tcp_w32_w", "tcp_red_w", "udp_reaper_w"]

print(f"Train: {TRAIN_SIZE:.0%}")
print(f"Validación: {VALID_SIZE:.0%}")
print(f"Test: {TEST_SIZE:.0%}")

Train: 70%
Validación: 15%
Test: 15%


In [21]:
# Revisión de estratos de particionamiento
query_strata_counts = f"""
WITH base AS (
    SELECT
        dataset_name,
        CASE
            WHEN attack_a = 0 THEN 'normal'
            ELSE attack_t
        END AS clase,
        COUNT(*) AS n
    FROM read_parquet('{parquet_sql_path}')
    GROUP BY dataset_name, clase
)
SELECT
    dataset_name, clase,
    dataset_name || '__' || clase AS strata_key,
    n
FROM base ORDER BY dataset_name, clase
"""
df_strata_counts = con.execute(query_strata_counts).df()
df_strata_counts

,dataset_name,clase,strata_key,n
0,BLASTER_WORM_v2,normal,BLASTER_WORM_v2__normal,3095187
1,BLASTER_WORM_v2,tcp_w32_w,BLASTER_WORM_v2__tcp_w32_w,24291
2,HTTP_FLOOD_v2,http_f,HTTP_FLOOD_v2__http_f,22959
3,HTTP_FLOOD_v2,normal,HTTP_FLOOD_v2__normal,4086158
4,ICMP_FLOOD_v2,icmp_f,ICMP_FLOOD_v2__icmp_f,11628
5,ICMP_FLOOD_v2,icmp_smf,ICMP_FLOOD_v2__icmp_smf,59479
6,ICMP_FLOOD_v2,normal,ICMP_FLOOD_v2__normal,4336095
7,REAPER_WORM_v2,normal,REAPER_WORM_v2__normal,4671854
8,REAPER_WORM_v2,udp_reaper_w,REAPER_WORM_v2__udp_reaper_w,1176
9,RED_WORM_v2,normal,RED_WORM_v2__normal,4381608


In [22]:
# Validación de tamaño mínimo por estrato
min_stratum_size = df_strata_counts["n"].min()
print(f"Número de estratos: {len(df_strata_counts)}")
print(f"Tamaño mínimo de estrato: {min_stratum_size:,}")
if min_stratum_size < 3:
    raise ValueError(
        "Hay estratos con menos de 3 registros. "
        "No se puede garantizar presencia en train, validación y test."
    )
df_strata_counts.sort_values("n")

Número de estratos: 13
Tamaño mínimo de estrato: 1,176


,dataset_name,clase,strata_key,n
8,REAPER_WORM_v2,udp_reaper_w,REAPER_WORM_v2__udp_reaper_w,1176
4,ICMP_FLOOD_v2,icmp_f,ICMP_FLOOD_v2__icmp_f,11628
2,HTTP_FLOOD_v2,http_f,HTTP_FLOOD_v2__http_f,22959
1,BLASTER_WORM_v2,tcp_w32_w,BLASTER_WORM_v2__tcp_w32_w,24291
5,ICMP_FLOOD_v2,icmp_smf,ICMP_FLOOD_v2__icmp_smf,59479
12,UDP_FLOOD_v2,udp_f,UDP_FLOOD_v2__udp_f,93583
11,UDP_FLOOD_v2,normal,UDP_FLOOD_v2__normal,536541
10,RED_WORM_v2,tcp_red_w,RED_WORM_v2__tcp_red_w,1255702
0,BLASTER_WORM_v2,normal,BLASTER_WORM_v2__normal,3095187
3,HTTP_FLOOD_v2,normal,HTTP_FLOOD_v2__normal,4086158


### Crear dataset con columna de split

Se crea un nuevo archivo en data/interim (versión intermedia para modelado) donde este archivo tendrá las 26 columnas originales del dataset preparado más cuatro columnas auxiliares de particionamiento: row_id_model, clase, strata_key, split

In [23]:
# Crear archivo con partición train / valid / test
import shutil
import gc
import time
split_dataset_path = (INTERIM_PATH / "dataset_modelado_con_split.parquet")
split_dataset_temp_path = (INTERIM_PATH / "dataset_modelado_con_split_tmp.parquet")
def quote_identifier(column_name):
    return '"' + column_name.replace('"', '""') + '"'

prepared_columns_sql = ",\n        ".join(
    quote_identifier(col)
    for col in prepared_columns
)
split_dataset_temp_sql_path = (split_dataset_temp_path
    .resolve()
    .as_posix()
    .replace("'", "''")
)
# Eliminar temporal previo, si existe
if split_dataset_temp_path.exists():
    split_dataset_temp_path.unlink()

query_create_split_dataset = f"""
COPY (
    WITH base AS (
        SELECT ROW_NUMBER() OVER () AS row_id_model, *,
            CASE
                WHEN attack_a = 0 THEN 'normal'
                ELSE attack_t
            END AS clase,
            dataset_name || '__' ||
            CASE
                WHEN attack_a = 0 THEN 'normal'
                ELSE attack_t
            END AS strata_key
        FROM read_parquet('{parquet_sql_path}')
    ),
    ranked AS (
        SELECT *,
            COUNT(*) OVER (
                PARTITION BY strata_key
            ) AS stratum_n,
            ROW_NUMBER() OVER (
                PARTITION BY strata_key
                ORDER BY hash(CAST(row_id_model AS VARCHAR) || '_{RANDOM_STATE}')
            ) AS rn FROM base
    ),
    assigned AS (
        SELECT *,
            CASE
                WHEN rn <= FLOOR(stratum_n * {TRAIN_SIZE})
                    THEN 'train'
                WHEN rn <= FLOOR(stratum_n * {TRAIN_SIZE + VALID_SIZE})
                    THEN 'valid'
                ELSE 'test'
            END AS split FROM ranked
    )
    SELECT row_id_model, {prepared_columns_sql}, clase, strata_key, split
    FROM assigned
)
TO '{split_dataset_temp_sql_path}'
(FORMAT PARQUET, COMPRESSION 'snappy')
"""
con.execute(query_create_split_dataset)
gc.collect()
time.sleep(1)
print("Archivo temporal con split creado:")
print(split_dataset_temp_path)

Archivo temporal con split creado:
C:\Users\Laura\Documents\TrabajoGrado2026\TG2026\data\interim\dataset_modelado_con_split_tmp.parquet


In [ ]:
# Validar y consolidar archivo con split - os replace 
split_temp_metadata = pq.read_metadata(split_dataset_temp_path)
split_temp_rows = split_temp_metadata.num_rows
split_temp_columns = split_temp_metadata.schema.names
del split_temp_metadata
gc.collect()
time.sleep(1)
print(f"Filas archivo preparado: {n_rows:,}")
print(f"Filas archivo con split: {split_temp_rows:,}")
print(f"Columnas archivo con split: {len(split_temp_columns)}")
if split_temp_rows != n_rows:
    raise ValueError(
        "El archivo con split no conserva el mismo número de filas que el dataset preparado.")

if split_dataset_path.exists():
    split_dataset_path.unlink()

os.replace(split_dataset_temp_path, split_dataset_path)
print("\nArchivo de modelado con split creado correctamente.")
print(f"Ruta: {split_dataset_path}")

In [29]:
# Validar distribución general de train / valid / test
split_dataset_sql_path = (
    split_dataset_path
    .resolve()
    .as_posix()
    .replace("'", "''")
)
query_split_distribution = f"""
SELECT split, COUNT(*) AS n,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER() AS porcentaje
FROM read_parquet('{split_dataset_sql_path}')
GROUP BY split
ORDER BY
    CASE split
        WHEN 'train' THEN 1
        WHEN 'valid' THEN 2
        WHEN 'test' THEN 3
        ELSE 4
    END
"""
df_split_distribution = con.execute(query_split_distribution).df()
df_split_distribution

,split,n,porcentaje
0,train,15803376,69.999970
1,valid,3386438,14.999995
2,test,3386447,15.000035


train ≈ 70 %
valid ≈ 15 %
test  ≈ 15 %

In [ ]:
# Validar distribución binaria por split
query_split_binary_distribution = f"""
SELECT split, attack_a, COUNT(*) AS n,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY split) AS porcentaje_en_split
FROM read_parquet('{split_dataset_sql_path}')
GROUP BY split, attack_a
ORDER BY
    CASE split
        WHEN 'train' THEN 1
        WHEN 'valid' THEN 2
        WHEN 'test' THEN 3
        ELSE 4
    END,
    attack_a
"""
df_split_binary_distribution = con.execute(query_split_binary_distribution).df()
df_split_binary_distribution

,split,attack_a,n,porcentaje_en_split
0,train,0,14775206,93.493985
1,train,1,1028170,6.506015
2,valid,0,3166116,93.493990
3,valid,1,220322,6.506010
4,test,0,3166121,93.493889
5,test,1,220326,6.506111


Esto debe confirmar que en train, validación y test se mantiene aproximadamente el mismo porcentaje de normal y ataque.

In [31]:
# Validar distribución por clase y split
query_split_class_distribution = f"""
SELECT split, clase, COUNT(*) AS n,
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY split) AS porcentaje_en_split
FROM read_parquet('{split_dataset_sql_path}')
GROUP BY split, clase
ORDER BY
    CASE split
        WHEN 'train' THEN 1
        WHEN 'valid' THEN 2
        WHEN 'test' THEN 3
        ELSE 4
    END,
    CASE clase
        WHEN 'normal' THEN 1
        WHEN 'icmp_f' THEN 2
        WHEN 'icmp_smf' THEN 3
        WHEN 'udp_f' THEN 4
        WHEN 'http_f' THEN 5
        WHEN 'tcp_w32_w' THEN 6
        WHEN 'tcp_red_w' THEN 7
        WHEN 'udp_reaper_w' THEN 8
        ELSE 9
    END
"""
df_split_class_distribution = con.execute(query_split_class_distribution).df()
df_split_class_distribution

,split,clase,n,porcentaje_en_split
0,train,normal,14775206,93.493985
1,train,icmp_f,8139,0.051502
2,train,icmp_smf,41635,0.263456
3,train,udp_f,65508,0.414519
4,train,http_f,16071,0.101693
5,train,tcp_w32_w,17003,0.107591
6,train,tcp_red_w,878991,5.562046
7,train,udp_reaper_w,823,0.005208
8,valid,normal,3166116,93.493990
9,valid,icmp_f,1744,0.051500


Esta es una de las salidas más importantes. Nos va a mostrar, por ejemplo, que udp_reaper_w quedó representado en train, validación y test.

In [32]:
# Validación de presencia de clases en todos los splits
query_class_presence_validation = f"""
SELECT clase,
    SUM(CASE WHEN split = 'train' THEN 1 ELSE 0 END) AS n_train,
    SUM(CASE WHEN split = 'valid' THEN 1 ELSE 0 END) AS n_valid,
    SUM(CASE WHEN split = 'test' THEN 1 ELSE 0 END) AS n_test
FROM read_parquet('{split_dataset_sql_path}')
GROUP BY clase
ORDER BY
    CASE clase
        WHEN 'normal' THEN 1
        WHEN 'icmp_f' THEN 2
        WHEN 'icmp_smf' THEN 3
        WHEN 'udp_f' THEN 4
        WHEN 'http_f' THEN 5
        WHEN 'tcp_w32_w' THEN 6
        WHEN 'tcp_red_w' THEN 7
        WHEN 'udp_reaper_w' THEN 8
        ELSE 9
    END
"""
df_class_presence_validation = con.execute(query_class_presence_validation).df()
df_class_presence_validation

,clase,n_train,n_valid,n_test
0,normal,14775206.0,3166116.0,3166121.0
1,icmp_f,8139.0,1744.0,1745.0
2,icmp_smf,41635.0,8922.0,8922.0
3,udp_f,65508.0,14037.0,14038.0
4,http_f,16071.0,3444.0,3444.0
5,tcp_w32_w,17003.0,3644.0,3644.0
6,tcp_red_w,878991.0,188355.0,188356.0
7,udp_reaper_w,823.0,176.0,177.0


In [ ]:
#eliminar este de aca abajo

In [ ]:
# ============================================================
# Validación automática de presencia de clases
class_presence_errors = df_class_presence_validation[
    (df_class_presence_validation["n_train"] == 0)
    | (df_class_presence_validation["n_valid"] == 0)
    | (df_class_presence_validation["n_test"] == 0)
]

if not class_presence_errors.empty:
    raise ValueError(
        "Hay clases que no quedaron representadas en todos los splits."
    )

print("Todas las clases están representadas en train, validación y test.")

Todas las clases están representadas en train, validación y test.


In [34]:
# Validación de estratos por dataset y clase
query_strata_split_validation = f"""
SELECT strata_key,
    SUM(CASE WHEN split = 'train' THEN 1 ELSE 0 END) AS n_train,
    SUM(CASE WHEN split = 'valid' THEN 1 ELSE 0 END) AS n_valid,
    SUM(CASE WHEN split = 'test' THEN 1 ELSE 0 END) AS n_test,
    COUNT(*) AS n_total
FROM read_parquet('{split_dataset_sql_path}')
GROUP BY strata_key
ORDER BY n_total ASC
"""
df_strata_split_validation = con.execute(query_strata_split_validation).df()
df_strata_split_validation

,strata_key,n_train,n_valid,n_test,n_total
0,REAPER_WORM_v2__udp_reaper_w,823.0,176.0,177.0,1176
1,ICMP_FLOOD_v2__icmp_f,8139.0,1744.0,1745.0,11628
2,HTTP_FLOOD_v2__http_f,16071.0,3444.0,3444.0,22959
3,BLASTER_WORM_v2__tcp_w32_w,17003.0,3644.0,3644.0,24291
4,ICMP_FLOOD_v2__icmp_smf,41635.0,8922.0,8922.0,59479
5,UDP_FLOOD_v2__udp_f,65508.0,14037.0,14038.0,93583
6,UDP_FLOOD_v2__normal,375578.0,80481.0,80482.0,536541
7,RED_WORM_v2__tcp_red_w,878991.0,188355.0,188356.0,1255702
8,BLASTER_WORM_v2__normal,2166630.0,464278.0,464279.0,3095187
9,HTTP_FLOOD_v2__normal,2860310.0,612924.0,612924.0,4086158


In [ ]:
#eliminar el de abajo

In [35]:
# Validación automática de estratos
strata_presence_errors = df_strata_split_validation[
    (df_strata_split_validation["n_train"] == 0)
    | (df_strata_split_validation["n_valid"] == 0)
    | (df_strata_split_validation["n_test"] == 0)
]
if not strata_presence_errors.empty:
    raise ValueError(
        "Hay estratos dataset-clase que no quedaron representados "
        "en todos los splits."
    )
print("Todos los estratos dataset-clase están representados en los tres splits.")

Todos los estratos dataset-clase están representados en los tres splits.


In [ ]:
# Guardar resúmenes del particionamiento
split_distribution_path = (PREPROCESSING_RESULTS_PATH / "split_distribucion_general.csv")
split_binary_distribution_path = (PREPROCESSING_RESULTS_PATH / "split_distribucion_binaria.csv")
split_class_distribution_path = (PREPROCESSING_RESULTS_PATH / "split_distribucion_por_clase.csv")
split_strata_validation_path = (PREPROCESSING_RESULTS_PATH / "split_validacion_estratos.csv")
df_split_distribution.to_csv(split_distribution_path, index=False, encoding="utf-8-sig")
df_split_binary_distribution.to_csv(split_binary_distribution_path, index=False, encoding="utf-8-sig")
df_split_class_distribution.to_csv(split_class_distribution_path, index=False, encoding="utf-8-sig")
df_strata_split_validation.to_csv(split_strata_validation_path, index=False, encoding="utf-8-sig")

print("Resúmenes de split guardados correctamente.")
print(split_distribution_path)
print(split_binary_distribution_path)
print(split_class_distribution_path)
print(split_strata_validation_path)

Se generó una única partición base del conjunto preparado en entrenamiento, validación y prueba, con proporciones aproximadas de 70 %, 15 % y 15 %, respectivamente. Para la tarea binaria se utilizarán registros normales y de ataque, mientras que para la clasificación del tipo de ataque se filtrarán únicamente los registros con `attack_a = 1`, conservando el mismo split.